# Tutorial: write `run.py` (baseline)

You already have:

- `helpers.py` — load CSVs, write the AIcrowd file
- `src/preprocess.py` — drop / sentinels / median / standardize
- `implementations.py` — `least_squares`, `logistic_regression`, …

This notebook **glues them**. Copy the last cells into **`ML_project_1/run.py`** (replace the shape-print script).

**Order (do not swap 2 and 3):**

```text
load raw x, y
 → shuffle-split train / val          # before any median
 → fit_preprocess on TRAIN only
 → transform val and test
 → add_bias
 → least_squares  and  logistic
 → map scores to {+1, -1}
 → print val metrics  (not train accuracy)
 → create_csv_submission(ids_test, y_hat, ...)
```

No pandas, no sklearn. Kernel **`ml`**. Use `data/cache_eda.npz` if it exists (fast).

`least_squares` is a **179 × 179** solve — seconds. Logistic is GD on ~260k rows — tens of seconds for ~50 steps; that is enough for this tutorial.


## 0. Paths

The notebook lives in `doc/`. `run.py` will live at the **repo root**, so it will import without these `Path` tricks.


In [1]:
import sys
from pathlib import Path

import numpy as np

DOC = Path(".").resolve()
ROOT = DOC.parent if DOC.name == "doc" else DOC
DATA = ROOT / "data"
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))

from helpers import load_csv_data, create_csv_submission
from implementations import least_squares, logistic_regression
from preprocess import (
    load_feature_names,
    fit_preprocess,
    transform_preprocess,
    add_bias,
    train_val_split,
)

print("ROOT", ROOT)


ROOT /Users/yijunliu/Documents/ML/ML_project_1


## 1. Load raw arrays

`y` stays `{+1, -1}` until logistic. Do **not** preprocess before the split.


In [ ]:
cache = DATA / "cache_eda.npz"
names = load_feature_names(DATA)

if cache.exists():
    z = np.load(cache)
    y, x, ids_tr = z["y"], z["x"], z["ids_tr"]
    x_te, ids_te = z["x_te"], z["ids_te"]
    print("cache", cache)
else:
    y, x, ids_tr, x_te, ids_te = load_csv_data(str(DATA))
    print("loaded CSVs (slow)")

print(x.shape, x_te.shape, y.shape)


## 2. Split **first**

`train_val_split` shuffles with a **fixed seed** so the val set does not change every run.

Medians must be computed on `x_tr` only. Val is a fake “test” you have labels for.


In [ ]:
x_tr, y_tr, id_tr, x_val, y_val, id_val = train_val_split(
    x, y, ids_tr, val_ratio=0.2, seed=0
)
print("train", x_tr.shape, "val", x_val.shape)
print("train P(+1)", float(np.mean(y_tr == 1)), "val P(+1)", float(np.mean(y_val == 1)))


## 3. Preprocess: fit on train, apply to val/test

`tx` = features **plus a column of ones** (intercept). Add the ones **after** scaling so you do not standardize the 1s.


In [ ]:
x_tr_z, stats = fit_preprocess(x_tr, names)
x_val_z = transform_preprocess(x_val, names, stats)
x_te_z = transform_preprocess(x_te, names, stats)

tx_tr = add_bias(x_tr_z)
tx_val = add_bias(x_val_z)
tx_te = add_bias(x_te_z)
print("tx_tr", tx_tr.shape, "nan", np.isnan(tx_tr).sum())


## 4. Metrics — do not trust accuracy

Always-predict `-1` is already ~91% “accurate”. We also print **recall** and **F1 of the +1 class**.


In [ ]:
def to_pm1(yhat):
    """Map scores or {0,1} to {+1,-1}. Zeros (rare) → -1."""
    yhat = np.sign(np.asarray(yhat).reshape(-1))
    yhat[yhat == 0] = -1
    return yhat.astype(int)


def metrics(y_true, y_pred, title=""):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = to_pm1(y_pred)
    acc = np.mean(y_true == y_pred)
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == -1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == -1))
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    print(
        f"{title:20s}  acc={acc:.4f}  rec+={rec:.4f}  prec+={prec:.4f}  F1+={f1:.4f}"
        f"  (tp={tp} fp={fp} fn={fn})"
    )
    return acc, rec, f1


# Dummy: always healthy
metrics(y_val, np.full_like(y_val, -1), "always -1")


## 5. Baseline A — `least_squares`

Labels stay `{+1, -1}`. Prediction: sign of `tx @ w`.


In [ ]:
w_ls, loss_ls = least_squares(y_tr, tx_tr)
print("train MSE", float(loss_ls), "w[:5]", w_ls[:5])

pred_val_ls = tx_val @ w_ls
metrics(y_val, pred_val_ls, "least_squares val")
metrics(y_tr, tx_tr @ w_ls, "least_squares train")


If **train** F1 is much higher than **val** F1, you overfit. If both equal the always-`-1` row, the model is not using the features yet.


## 6. Baseline B — `logistic_regression`

The staff function needs `y ∈ {0, 1}`:

```text
y01 = (y_pm1 + 1) / 2     #  -1 → 0,  +1 → 1
```

Probability `σ(tx @ w)`; class `+1` if `p ≥ 0.5`.

Start with `max_iters=50`, `gamma=0.1`, `initial_w=0`. You can raise iters later.


In [ ]:
y_tr01 = (y_tr + 1) / 2.0
w0 = np.zeros(tx_tr.shape[1])
w_log, loss_log = logistic_regression(
    y_tr01, tx_tr, w0, max_iters=50, gamma=0.1
)
print("train NLL", float(loss_log))

p_val = 1.0 / (1.0 + np.exp(-np.clip(tx_val @ w_log, -30, 30)))
pred_val_log = np.where(p_val >= 0.5, 1, -1)
metrics(y_val, pred_val_log, "logistic val")


## 7. Choose one model for the CSV

For the first AIcrowd file, pick the **higher val F1**. Then predict **test** with that `w` (test has no `y`).


In [ ]:
# default: least squares (closed form, stable). Switch to w_log if F1 was better.
w_sub = w_ls
scores_te = tx_te @ w_sub
y_te_hat = to_pm1(scores_te)
print("test pred unique", np.unique(y_te_hat, return_counts=True))
print("test P(+1)", float(np.mean(y_te_hat == 1)))


## 8. Write the submission file

Header must be `Id,Prediction`. Ids = **`ids_te`**, same order as `x_test.csv`.


In [ ]:
out_path = ROOT / "submission_baseline.csv"
create_csv_submission(ids_te, y_te_hat, str(out_path))
with open(out_path) as f:
    for i, line in enumerate(f):
        print(line.strip())
        if i >= 4:
            break
print("wrote", out_path, "n lines", 1 + len(y_te_hat))


Do **not** git-commit this CSV if it is huge and regenerated by `run.py`. Upload it on AIcrowd once the team exists.

## 9. Copy into `run.py`

At the **repo root**, `run.py` should:

```python
from helpers import load_csv_data, create_csv_submission
from implementations import least_squares, logistic_regression
import sys
sys.path.insert(0, "src")
from preprocess import (
    load_feature_names, fit_preprocess, transform_preprocess,
    add_bias, train_val_split,
)
```

Then the same steps as cells 1–8, with `load_csv_data("data")` (no `doc/` paths). Optionally skip the cache.

**Checklist**

- split before `fit_preprocess`
- logistic `y` mapped to `{0,1}` only inside the logistic call
- submission `{+1,-1}` ints, ids from **test**
- print val metrics vs `always -1`

Next after this baseline: ridge / more logistic iters / class imbalance — **one change at a time**.
